# 01 — Model Exploration

Explores the SLM model architecture — instantiation, parameter counts, forward pass, attention patterns, and memory estimates.

**No GPU required** — everything here runs on CPU.

In [ ]:
import sys
sys.path.insert(0, '..')  # add repo root to path

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
from model import SLMConfig, SLMForCausalLM, SLM_125M, SLM_350M, SLM_1B, CONFIGS

## 1. Instantiate the 125M model

In [ ]:
config = SLM_125M
model = SLMForCausalLM(config)
model.eval()
print(model)

## 2. Parameter counts

In [ ]:
def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

total, trainable = count_params(model)
print(f"Total parameters:     {total:>15,}")
print(f"Trainable parameters: {trainable:>15,}")
print(f"Non-trainable:        {total - trainable:>15,}")

## 3. Parameter breakdown by component

In [ ]:
def param_breakdown(model):
    components = {
        "embed_tokens": 0,
        "attention (q/k/v/o)": 0,
        "mlp (gate/up/down)": 0,
        "norms": 0,
        "lm_head": 0,
    }
    for name, param in model.named_parameters():
        n = param.numel()
        if "embed_tokens" in name:
            components["embed_tokens"] += n
        elif any(k in name for k in ["q_proj", "k_proj", "v_proj", "o_proj"]):
            components["attention (q/k/v/o)"] += n
        elif any(k in name for k in ["gate_proj", "up_proj", "down_proj"]):
            components["mlp (gate/up/down)"] += n
        elif "norm" in name:
            components["norms"] += n
        elif "lm_head" in name:
            components["lm_head"] += n
    return components

breakdown = param_breakdown(model)
total_params = sum(breakdown.values())

df = pd.DataFrame([
    {"Component": k, "Parameters": v, "% of total": f"{100 * v / total_params:.1f}%"}
    for k, v in breakdown.items()
])
df["Parameters"] = df["Parameters"].apply(lambda x: f"{x:,}")
print(df.to_string(index=False))

In [ ]:
# Pie chart
fig, ax = plt.subplots(figsize=(7, 5))
labels = list(breakdown.keys())
sizes = list(breakdown.values())
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"]
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, autopct="%1.1f%%",
    colors=colors, startangle=140,
    wedgeprops={"edgecolor": "white", "linewidth": 1.5}
)
for t in autotexts:
    t.set_fontsize(9)
ax.set_title("SLM-125M — Parameter Distribution", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Compare all three model sizes

In [ ]:
rows = []
for size, cfg in CONFIGS.items():
    m = SLMForCausalLM(cfg)
    total, _ = count_params(m)
    rows.append({
        "Model": f"slm-{size}",
        "Layers": cfg.num_hidden_layers,
        "Hidden": cfg.hidden_size,
        "Intermediate": cfg.intermediate_size,
        "Q heads": cfg.num_attention_heads,
        "KV heads": cfg.num_key_value_heads,
        "Context": cfg.max_position_embeddings,
        "Parameters": f"{total / 1e6:.1f}M",
    })
    del m

df = pd.DataFrame(rows)
print(df.to_string(index=False))

## 5. Forward pass

In [ ]:
torch.manual_seed(42)
batch_size, seq_len = 2, 128
input_ids = torch.randint(0, config.vocab_size, (batch_size, seq_len))
labels = input_ids.clone()

with torch.no_grad():
    output = model(input_ids=input_ids, labels=labels)

print(f"Loss:           {output.loss.item():.4f}")
print(f"Perplexity:     {torch.exp(output.loss).item():.2f}")
print(f"Logits shape:   {output.logits.shape}")
print(f"Expected loss (random init ≈ log(vocab_size)): {torch.log(torch.tensor(config.vocab_size)).item():.4f}")

A randomly initialized model should produce a loss close to `log(vocab_size)` — this confirms the model and loss function are working correctly.

## 6. KV cache memory estimate

In [ ]:
def kv_cache_mb(cfg, seq_len, batch_size=1, dtype_bytes=2):
    """
    Estimate KV cache memory in MB.
    
    KV cache = 2 (K and V) * num_layers * num_kv_heads * seq_len * head_dim * dtype_bytes
    """
    head_dim = cfg.hidden_size // cfg.num_attention_heads
    bytes_per_token = 2 * cfg.num_hidden_layers * cfg.num_key_value_heads * head_dim * dtype_bytes
    total_bytes = bytes_per_token * seq_len * batch_size
    return total_bytes / (1024 ** 2)

print("KV Cache Memory Estimates (bfloat16, batch_size=1)")
print("-" * 55)
for size, cfg in CONFIGS.items():
    for seq in [512, 1024, 2048, 4096]:
        if seq <= cfg.max_position_embeddings:
            mb = kv_cache_mb(cfg, seq)
            print(f"  slm-{size:<5} | seq={seq:<5} | {mb:.1f} MB")

## 7. Attention pattern visualization (single layer)

In [ ]:
# Hook to capture attention weights from layer 0
attention_weights = {}

def make_hook(layer_idx):
    def hook(module, input, output):
        # Compute attention scores manually for visualization
        hidden = input[0]  # (batch, seq, hidden)
        bsz, seq, _ = hidden.shape
        q = module.q_proj(hidden)
        k = module.k_proj(hidden)
        q = q.view(bsz, seq, module.num_heads, module.head_dim).transpose(1, 2)
        k = k.view(bsz, seq, module.num_kv_heads, module.head_dim).transpose(1, 2)
        # Expand KV for GQA
        if module.num_kv_heads != module.num_heads:
            k = k.repeat_interleave(module.num_query_groups, dim=1)
        scale = module.head_dim ** -0.5
        scores = torch.softmax(
            torch.matmul(q, k.transpose(-2, -1)) * scale +
            torch.triu(torch.full((seq, seq), float('-inf')), diagonal=1),
            dim=-1
        )
        attention_weights[layer_idx] = scores.detach().cpu()
    return hook

hook = model.model.layers[0].self_attn.register_forward_hook(make_hook(0))

seq_len = 32
input_ids_vis = torch.randint(0, config.vocab_size, (1, seq_len))
with torch.no_grad():
    _ = model(input_ids=input_ids_vis)

hook.remove()

# Plot first 4 attention heads from layer 0
attn = attention_weights[0][0]  # (num_heads, seq, seq)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, ax in enumerate(axes):
    im = ax.imshow(attn[i].numpy(), cmap="Blues", aspect="auto", vmin=0, vmax=1)
    ax.set_title(f"Head {i}", fontsize=11)
    ax.set_xlabel("Key position")
    ax.set_ylabel("Query position")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle("SLM-125M — Layer 0 Attention Patterns (causal mask visible)", 
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

The upper triangle should be zero (causal masking — future tokens are not attended to). The lower triangle shows the attention distribution over past tokens.

## 8. Tied embeddings verification

In [ ]:
embed_weight = model.model.embed_tokens.weight
lm_head_weight = model.lm_head.weight

are_same = embed_weight.data_ptr() == lm_head_weight.data_ptr()
print(f"Embed tokens weight ptr:  {embed_weight.data_ptr()}")
print(f"LM head weight ptr:       {lm_head_weight.data_ptr()}")
print(f"Weights are tied:         {are_same}")
print(f"Weight shape:             {embed_weight.shape}")

## 9. RoPE frequency visualization

In [ ]:
rope = model.model.layers[0].self_attn.rotary_emb
cos, sin = rope(config.max_position_embeddings)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

im0 = axes[0].imshow(cos.numpy(), aspect="auto", cmap="RdBu", vmin=-1, vmax=1)
axes[0].set_title("RoPE cos — position × dimension", fontsize=11)
axes[0].set_xlabel("Head dimension")
axes[0].set_ylabel("Position")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(sin.numpy(), aspect="auto", cmap="RdBu", vmin=-1, vmax=1)
axes[1].set_title("RoPE sin — position × dimension", fontsize=11)
axes[1].set_xlabel("Head dimension")
axes[1].set_ylabel("Position")
plt.colorbar(im1, ax=axes[1])

fig.suptitle("Rotary Position Embeddings — cos/sin cache", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"cos/sin cache shape: {cos.shape}  (max_position_embeddings × head_dim)")
print(f"RoPE base (theta):   {config.rope_theta}")